In [ ]:
%pip install requests beautifulsoup4 urllib3

In [22]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
from collections import deque
import urllib3

# Configuration
BASE_URL = "https://mnit.ac.in"

# Determine download directory
# Assuming the notebook is running in RAG/pdfs_scrapping, we want to save to ./pdfs
# If running from root, we might want RAG/pdfs_scrapping/pdfs
if os.path.basename(os.getcwd()) == "pdfs_scrapping":
    DOWNLOAD_DIR = os.path.join(os.getcwd(), "pdfs")
else:
    DOWNLOAD_DIR = os.path.join(os.getcwd(), "RAG", "pdfs_scrapping", "pdfs")

if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)
    
print(f"Download directory: {DOWNLOAD_DIR}")

VISITED_PAGES = set()
PDF_LINKS = set()
PDF_SOURCES = {} # Map pdf_url -> source_page_url

# Set of extensions to ignore when looking for new pages to crawl
IGNORE_EXTENSIONS = {
    '.jpg', '.jpeg', '.png', '.gif', '.bmp', '.svg', '.webp',
    '.zip', '.rar', '.tar', '.gz', '.7z',
    '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx',
    '.mp3', '.mp4', '.avi', '.mov',
    '.css', '.js', '.json', '.xml'
}

Download directory: /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs


In [23]:
def is_valid_url(url):
    """Check if the URL belongs to the mnit.ac.in domain."""
    try:
        parsed = urlparse(url)
        # Allow mnit.ac.in and its subdomains
        return parsed.netloc.endswith("mnit.ac.in")
    except:
        return False

def is_pdf(url):
    """Check if the URL points to a PDF file."""
    path = urlparse(url).path
    return path.lower().endswith('.pdf')

def should_crawl(url):
    """Determine if we should crawl this URL for more links."""
    if not is_valid_url(url):
        return False
    
    path = urlparse(url).path.lower()
    if any(path.endswith(ext) for ext in IGNORE_EXTENSIONS):
        return False
        
    return True

In [24]:
def download_pdf(url, source_url=None):
    """Download a PDF file."""
    try:
        if url in PDF_LINKS:
            return
        
        PDF_LINKS.add(url)
        if source_url:
            PDF_SOURCES[url] = source_url
        
        # Create filename from URL
        parsed_url = urlparse(url)
        filename = os.path.basename(parsed_url.path)
        
        # Decode URL encoding in filename (e.g. %20 -> space)
        from urllib.parse import unquote
        filename = unquote(filename)
        
        if not filename.lower().endswith('.pdf'):
            filename += '.pdf'
            
        # Clean filename
        filename = "".join([c for c in filename if c.isalpha() or c.isdigit() or c in (' ', '.', '_', '-')]).strip()
        if not filename:
            filename = "document.pdf"

        # Handle duplicate filenames
        filepath = os.path.join(DOWNLOAD_DIR, filename)
        counter = 1
        while os.path.exists(filepath):
            name, ext = os.path.splitext(filename)
            filepath = os.path.join(DOWNLOAD_DIR, f"{name}_{counter}{ext}")
            counter += 1
            
        print(f"Downloading PDF: {url}")
        
        # Use a session for connection pooling
        response = requests.get(url, stream=True, timeout=30, verify=False) # verify=False because some academic sites have cert issues
        
        if response.status_code == 200:
            content_type = response.headers.get('content-type', '').lower()
            if 'application/pdf' in content_type or url.lower().endswith('.pdf'):
                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Saved to {filepath}")
            else:
                print(f"Skipping {url} - Content-Type is {content_type}")
        else:
            print(f"Failed to download PDF {url}: Status {response.status_code}")
            
    except Exception as e:
        print(f"Error downloading {url}: {e}")

In [25]:
def get_links_from_page(url, session):
    """Extract all links from a webpage."""
    try:
        response = session.get(url, timeout=15, verify=False)
        if response.status_code != 200:
            print(f"Failed to retrieve page {url}: Status {response.status_code}")
            return []
            
        # Check content type to ensure it's HTML
        content_type = response.headers.get('content-type', '').lower()
        if 'text/html' not in content_type:
            return []

        soup = BeautifulSoup(response.content, "html.parser")
        links = set()
        
        for a_tag in soup.find_all("a", href=True):
            href = a_tag.attrs["href"]
            full_url = urljoin(url, href)
            
            # Remove fragments
            full_url = full_url.split("#")[0]
            
            if full_url:
                links.add(full_url)
                
        return links
    except Exception as e:
        print(f"Error processing page {url}: {e}")
        return []

In [ ]:
def start_crawling(start_url, max_downloads=None):
    # Suppress SSL warnings
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    })

    # Initialize queue with start_url
    queue = deque([start_url])
    
    # Track seen URLs to avoid O(N) lookups in queue
    # Initialize with already visited pages to avoid re-crawling if function is re-run
    seen_urls = set(VISITED_PAGES)
    seen_urls.add(start_url)
    
    print(f"Starting crawl from {start_url}")
    if max_downloads:
        print(f"Max downloads set to: {max_downloads}")
    
    while queue:
        # Check limit
        if max_downloads is not None and len(PDF_LINKS) >= max_downloads:
            print(f"Reached limit of {max_downloads} downloads. Stopping.")
            break

        current_url = queue.popleft()
        
        if current_url in VISITED_PAGES:
            continue
            
        VISITED_PAGES.add(current_url)
        
        # If it's a PDF, download it (in case it got into the queue)
        if is_pdf(current_url):
            download_pdf(current_url, source_url="Queue/Unknown")
            continue
            
        # If we shouldn't crawl this page (external domain, image, etc), skip
        if not should_crawl(current_url):
            continue

        print(f"Scanning page: {current_url} | Queue size: {len(queue)} | PDFs found: {len(PDF_LINKS)}")
        
        new_links = get_links_from_page(current_url, session)
        
        for link in new_links:
            # Check limit before eager download
            if max_downloads is not None and len(PDF_LINKS) >= max_downloads:
                print(f"Reached limit of {max_downloads} downloads. Stopping.")
                return

            # Eagerly download PDFs so we don't wait for BFS to reach them
            if is_pdf(link):
                download_pdf(link, source_url=current_url)
                continue
            
            # Add to queue if not seen before
            if link not in seen_urls:
                seen_urls.add(link)
                queue.append(link)

In [ ]:
# Reset visited pages to ensure we can restart if previously interrupted
VISITED_PAGES = set() 
print("Reset visited pages tracking.")

# Start the crawling process
# Set max_downloads to a number (e.g., 100) or None for all
start_crawling(BASE_URL, max_downloads=100)

Reset visited pages tracking.
Starting crawl from https://mnit.ac.in
Scanning page: https://mnit.ac.in | Queue size: 0 | PDFs found: 0
Scanning page: https://mnit.ac.in/academics/office_order.php | Queue size: 172 | PDFs found: 0
Scanning page: https://mnit.ac.in/academics/office_order.php | Queue size: 172 | PDFs found: 0
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/Institute_Assistantship.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/Institute_Assistantship.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/999.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/999.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/penalty_fee_challan.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/penalty_fee_challan.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/NOC_to_staff.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs/NOC_to_staff.pdf
Saved to /home/rishabh/coding/pro/RAG/pdfs_scrapping/pdfs

KeyboardInterrupt: 